# Aeris Time Offset Sync
Cross-correlate Aeris CH₄ against Picarro CH₄ to generate offset JSON files.

**Workflow:**
1. Configure paths and select a date
2. Auto cross-correlation suggests lag for each Aeris file
3. Visual confirmation with interactive slider
4. Save confirmed offsets to JSON

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.widgets as widgets
from pathlib import Path

%matplotlib widget

## Config — edit these paths

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
PICARRO_DIR   = Path('WYO_picarro_cleaned')
ULTRA460_DIR  = Path('WYO_aerisultra460_cleaned')
ULTRA321_DIR  = Path('LANL_aerisultra321_cleaned')

ULTRA460_JSON = Path('aeris460_timeshift.json')
ULTRA321_JSON = Path('aeris321_timeshift.json')

# ── Resample grid for cross-correlation (seconds) ─────────────────────────
RESAMPLE_S = 1

# ── Max lag to search (seconds) ───────────────────────────────────────────
MAX_LAG_S = 120

## Helpers

In [ ]:
def load_cleaned(path: Path) -> pd.Series:
    """Load a cleaned CSV and return CH4 as a DatetimeIndex Series."""
    df = pd.read_csv(path, index_col='TIMESTAMP', parse_dates=True)
    return df['CH4'].dropna()


def resample_series(s: pd.Series, freq_s: int) -> pd.Series:
    """Resample to a regular grid using mean, interpolate small gaps."""
    s = s.resample(f'{freq_s}s').mean()
    s = s.interpolate(method='time', limit=10)
    return s


def cross_correlate(ref: pd.Series, sig: pd.Series, max_lag_s: int, freq_s: int) -> float:
    """
    Cross-correlate sig against ref on their overlapping window.
    Returns the lag in seconds that maximises correlation.
    Positive lag  → sig is LATE  (shift sig backward, i.e. negative offset applied to sig)
    Negative lag  → sig is EARLY (shift sig forward)
    """
    # Overlap window
    start = max(ref.index[0], sig.index[0])
    end   = min(ref.index[-1], sig.index[-1])
    if start >= end:
        print('  [warn] No overlapping time window between ref and signal.')
        return 0.0

    r = ref[start:end].copy()
    s = sig[start:end].copy()

    # Align to same index
    combined = pd.DataFrame({'r': r, 's': s}).dropna()
    if len(combined) < 10:
        print('  [warn] Fewer than 10 overlapping points — skipping cross-correlation.')
        return 0.0

    r_arr = (combined['r'] - combined['r'].mean()).values
    s_arr = (combined['s'] - combined['s'].mean()).values

    max_lag_samples = max_lag_s // freq_s
    corr = np.correlate(r_arr, s_arr, mode='full')
    lags = np.arange(-(len(r_arr) - 1), len(r_arr))

    # Restrict to search window
    mask = np.abs(lags) <= max_lag_samples
    best_lag_samples = lags[mask][np.argmax(corr[mask])]
    best_lag_s = float(best_lag_samples * freq_s)
    return best_lag_s


def files_for_date(directory: Path, date_str: str) -> list[Path]:
    """
    Return all cleaned CSVs in directory whose filename contains date_str.
    date_str format: 'YYMMDD'  e.g. '260203'
    """
    return sorted(directory.glob(f'*{date_str}*_clean.csv'))


def picarro_for_date(directory: Path, date_str: str) -> pd.Series | None:
    """
    Load and concatenate all Picarro files for a given YYMMDD date.
    Returns a resampled CH4 Series or None if no files found.
    """
    files = sorted(directory.glob(f'*{date_str}*_clean.csv'))
    if not files:
        print(f'  [warn] No Picarro files found for date {date_str}')
        return None
    parts = [load_cleaned(f) for f in files]
    combined = pd.concat(parts).sort_index()
    combined = combined[~combined.index.duplicated(keep='first')]
    return resample_series(combined, RESAMPLE_S)


def load_json(path: Path) -> dict:
    with open(path) as f:
        return json.load(f)


def save_json(data: dict, path: Path):
    with open(path, 'w') as f:
        json.dump(data, f, indent=2)
    print(f'  ✓  Saved → {path}')


def json_key(cleaned_path: Path) -> str:
    """Convert cleaned filename back to original .txt key used in the JSON."""
    return cleaned_path.name.replace('_clean.csv', '.txt')


print('Helpers loaded.')

## Step 1 — Select date and instrument

Set `DATE` to the `YYMMDD` string you want to work on (matches the filename pattern).  
Set `INSTRUMENT` to `'460'` or `'321'`.

In [ ]:
DATE       = '260203'   # YYMMDD
INSTRUMENT = '321'      # '460' or '321'

# ── Resolve paths from config ──────────────────────────────────────────────
if INSTRUMENT == '460':
    aeris_dir  = ULTRA460_DIR
    offset_json = ULTRA460_JSON
elif INSTRUMENT == '321':
    aeris_dir  = ULTRA321_DIR
    offset_json = ULTRA321_JSON
else:
    raise ValueError(f"INSTRUMENT must be '460' or '321', got '{INSTRUMENT}'")

aeris_files = files_for_date(aeris_dir, DATE)
print(f'Instrument : Ultra {INSTRUMENT}')
print(f'Date       : {DATE}')
print(f'Aeris files found ({len(aeris_files)}):')
for f in aeris_files:
    print(f'  {f.name}')

picarro_ch4 = picarro_for_date(PICARRO_DIR, DATE)
if picarro_ch4 is not None:
    print(f'\nPicarro: {len(picarro_ch4)} samples  '
          f'({picarro_ch4.index[0]}  →  {picarro_ch4.index[-1]})')
else:
    print('\n[error] No Picarro data — cannot proceed.')

## Step 2 — Auto cross-correlation

Runs cross-correlation for every Aeris file on this date against the Picarro reference.  
Results are stored in `suggestions` dict: `{ filename: lag_seconds }`.

In [ ]:
suggestions = {}   # filename → suggested lag (seconds)

if picarro_ch4 is None:
    print('[error] No Picarro data loaded — run Step 1 first.')
else:
    for f in aeris_files:
        key = json_key(f)
        aeris_ch4 = resample_series(load_cleaned(f), RESAMPLE_S)
        lag = cross_correlate(picarro_ch4, aeris_ch4, MAX_LAG_S, RESAMPLE_S)
        suggestions[key] = lag
        print(f'  {f.name:<50}  suggested lag: {lag:+.0f}s')

## Step 3 — Visual confirmation

Pick one file to inspect. The plot shows:
- **Grey** — Picarro (reference)
- **Colour** — Aeris with current shift applied

Use the slider to nudge the offset. When it looks right, run Step 4 to commit.

In [ ]:
# ── Select which file to inspect ──────────────────────────────────────────
# Change the index to inspect a different file (0-based)
FILE_IDX = 0

# ── Load ──────────────────────────────────────────────────────────────────
inspect_file = aeris_files[FILE_IDX]
inspect_key  = json_key(inspect_file)
initial_lag  = suggestions.get(inspect_key, 0.0)

aeris_raw   = load_cleaned(inspect_file)
aeris_rs    = resample_series(aeris_raw, RESAMPLE_S)

# Overlap window for display
t_start = max(picarro_ch4.index[0], aeris_rs.index[0])
t_end   = min(picarro_ch4.index[-1], aeris_rs.index[-1])
pic_win  = picarro_ch4[t_start:t_end]
aer_win  = aeris_rs[t_start:t_end]

print(f'Inspecting : {inspect_file.name}')
print(f'Suggested lag : {initial_lag:+.0f}s')
print(f'Overlap window: {t_start}  →  {t_end}  ({len(pic_win)} samples)')

# ── Plot ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
plt.subplots_adjust(bottom=0.22)

line_pic, = ax.plot(pic_win.index, pic_win.values,
                    color='#aaaaaa', lw=1.2, label='Picarro (ref)')

# Apply initial shift
shifted_index = aer_win.index + pd.to_timedelta(initial_lag, unit='s')
line_aer, = ax.plot(shifted_index, aer_win.values,
                    color='#60a5fa', lw=1.2, label=f'Aeris {INSTRUMENT} (shifted)')

ax.set_xlabel('Time (UTC)')
ax.set_ylabel('CH₄')
ax.set_title(f'{inspect_file.name}  |  lag = {initial_lag:+.0f}s')
ax.legend(loc='upper right', fontsize=9)
fig.autofmt_xdate()

# ── Slider ────────────────────────────────────────────────────────────────
ax_slider = plt.axes([0.15, 0.06, 0.70, 0.04])
slider = widgets.Slider(
    ax_slider, 'Lag (s)',
    -MAX_LAG_S, MAX_LAG_S,
    valinit=initial_lag, valstep=1.0
)

def update(val):
    lag = slider.val
    shifted = aer_win.index + pd.to_timedelta(lag, unit='s')
    line_aer.set_xdata(shifted)
    ax.set_title(f'{inspect_file.name}  |  lag = {lag:+.0f}s')
    fig.canvas.draw_idle()

slider.on_changed(update)
plt.show()

print("\nWhen you're happy with the lag, run Step 4 to commit it.")

## Step 4 — Commit offset for this file

Reads the current slider value and writes it into the in-memory `confirmed` dict.  
Run this cell after you're happy with each file's slider.

In [ ]:
# confirmed offsets accumulate here across all files for this date
if 'confirmed' not in dir():
    confirmed = {}

lag_value = float(slider.val)
confirmed[inspect_key] = lag_value

print(f'  Committed  {inspect_key}  →  {lag_value:+.0f}s')
print(f'  confirmed so far: {confirmed}')

## Step 5 — Save to JSON

Merges confirmed offsets into the existing JSON file (preserves other dates/files already in there).

In [ ]:
existing = load_json(offset_json)
existing.update(confirmed)
save_json(existing, offset_json)

print(f'\nFull JSON after update:')
for k, v in existing.items():
    marker = '  ← updated' if k in confirmed else ''
    print(f'  {k:<55} {v:+.0f}s{marker}')